# **Deep Neural Network**

# **Importing the Dataset**

In [1]:
import torch
import torch.nn as nn
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

# **Load and create Dataframe**

In [2]:
cancer = load_breast_cancer()

df = pd.DataFrame(cancer.data,
    columns=cancer.feature_names)
df['Target'] = cancer.target

x = df.drop('Target', axis=1)
y = df['Target']

# **Split the Train & Test Split**

In [3]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# **Feature Scaling**

In [4]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# **Converting to Tensors**

In [5]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

In [6]:
print(x_train.shape)
print(y_train.shape)

print(df.head())

print(df["Target"].value_counts())

torch.Size([455, 30])
torch.Size([455, 1])
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  wo

# **Model Building**

In [7]:
class BreastCancerModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(30, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

  def forward(self, x):
    return self.network(x)

model = BreastCancerModel()


# **Loss Function**

In [8]:
criterion = nn.BCEWithLogitsLoss()

# **Optimizer**

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# **Converting Logits into Probabilities**

In [10]:
probabilities = torch.sigmoid(model(x_train))

# **Converting Probabilities into Classes**

In [11]:
predicted_classes = (probabilities >= 0.5).float()

# **Training Loop**

In [12]:
epochs = 1000

for epoch in range(epochs):

  logits = model(x_train)

  loss = criterion(logits, y_train)

  loss.backward()

  optimizer.step()

  optimizer.zero_grad()

  if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss.item():.4f}")

Epoch [100/1000] Loss: 0.5849
Epoch [200/1000] Loss: 0.4287
Epoch [300/1000] Loss: 0.2982
Epoch [400/1000] Loss: 0.2140
Epoch [500/1000] Loss: 0.1660
Epoch [600/1000] Loss: 0.1379
Epoch [700/1000] Loss: 0.1199
Epoch [800/1000] Loss: 0.1076
Epoch [900/1000] Loss: 0.0986
Epoch [1000/1000] Loss: 0.0917


# **Evaluation** & Accuracy Metrics

In [13]:
model.eval()

with torch.no_grad():
  logits = model(x_test)

  probabilities = torch.sigmoid(logits)

  predictions = (probabilities >= 0.5).float()

  # Accuracy Calculation

correct = (predictions == y_test).sum().item()

accuracy = correct / len(y_test)

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 97.37%
